In [ ]:
! git clone https://github.com/vrdn-23/SemEval-2020

In [ ]:
a = open("/content/SemEval-2020/datasets/train-articles/article111111111.txt", 'r').read()
a[265:323]
a[1795:1935]
a[149:157]
a[1069:1091]

'a very, very different'

In [ ]:
def load_spans(text):
  text = text.strip().split('\n')
  # print(text)
  output = [{"loc": (int(start), int(end)),
             "propType": sorted(pType.split(','))} for (article, pType, start, end) in sorted(list(map(lambda x: x.split('\t'), text)), key= lambda x: int(x[2]))] if text[0] else []
  return output

In [ ]:
from pathlib import Path

folder = Path("/content/SemEval-2020/datasets")
texts_folder = Path("/content/SemEval-2020/datasets/train-articles")
spans_folder = Path("/content/SemEval-2020/datasets/train-labels-task2-technique-classification")

dataset = []

for text_file in texts_folder.glob('*'):
    span_file = spans_folder / (text_file.stem + ".task2-TC.labels")

    text = text_file.read_text(encoding="utf-8")
    spans = load_spans(span_file.read_text(encoding="utf-8"))

    dataset.append({
        "text": text,
        "spans": spans
    })

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "roberta-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

PROPAGANDA_CLASSES = [
    'Appeal_to_Authority', 'Appeal_to_fear-prejudice', 'Bandwagon',
    'Black-and-White_Fallacy', 'Causal_Oversimplification', 'Doubt',
    'Exaggeration', 'Flag-Waving', 'Labeling', 'Loaded_Language',
    'Red_Herring', 'Repetition', 'Slogans', 'Thought-terminating_Cliches'
]

label_to_id = {name: i for i, name in enumerate(PROPAGANDA_CLASSES)}
id_to_label = {i: name for name, i in label_to_id.items()}

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=14,
    label2id=label_to_id,
    id2label=id_to_label
).to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        example["span"],
        padding=False,
        truncation=True,
        max_length=512
    )

    tokens["labels"] = label_to_id[example["label"]]

    return tokens

columns_to_remove = dataset["train"].column_names

train_dataset_tokenized = dataset["train"].map(
    tokenize_function,
    remove_columns=columns_to_remove,
    desc="Tokenizing train"
)

val_dataset_tokenized = dataset["validation"].map(
    tokenize_function,
    remove_columns=columns_to_remove,
    desc="Tokenizing val"
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DataCollatorWithPadding

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predicted_labels = np.argmax(predictions, axis=-1)

    accuracy = accuracy_score(labels, predicted_labels)
    P, R, F1, _ = precision_recall_fscore_support(
        labels, predicted_labels, average='macro', zero_division=0
    )

    return {'accuracy': accuracy, 'macro_precision': P, 'macro_recall': R, 'macro_f1': F1}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./roberta-propaganda-detector",
    eval_strategy="steps",
    save_strategy='steps',
    eval_steps=150,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    # load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

trainer = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

print(trainer.evaluate())

Step,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
150,No log,1.938400,0.405383,0.052022,0.127800,0.072807
300,No log,1.656983,0.468189,0.197538,0.172732,0.135540
450,No log,1.399934,0.578303,0.310578,0.289858,0.282183
600,1.835050,1.168325,0.637847,0.456776,0.432454,0.404006
750,1.835050,1.060195,0.671289,0.452589,0.417459,0.406401
900,1.835050,1.065157,0.685155,0.606415,0.486953,0.486520
1050,1.070173,0.964377,0.686786,0.507455,0.536353,0.499489
1200,1.070173,0.976439,0.703100,0.543068,0.533239,0.518925
1350,1.070173,1.001532,0.707178,0.544982,0.555202,0.530295
1500,0.778439,1.023732,0.695759,0.519857,0.541599,0.523092


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.3068063259124756, 'eval_accuracy': 0.7267536704730831, 'eval_macro_precision': 0.5831813016304748, 'eval_macro_recall': 0.6066635857907441, 'eval_macro_f1': 0.5920337954434078, 'eval_runtime': 37.4041, 'eval_samples_per_second': 32.777, 'eval_steps_per_second': 4.117, 'epoch': 5.0}
